# dashboard_failure_modes_v1

目标：验证 dashboard 可以从 manifest-first analysis report 读取 `failure_modes`，在 view model 和静态页面中展示最高优先级失败模式。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dashboard import build_dashboard_view_model, load_model_matrix_dashboard, write_dashboard_html

CARD = REPO_ROOT / 'research/dashboard_failure_modes_v1'
SOURCE_CARD = REPO_ROOT / 'research/analysis_failure_modes_v1/analysis_report'
CARD.mkdir(parents=True, exist_ok=True)
{'card': str(CARD.relative_to(REPO_ROOT)), 'source': str(SOURCE_CARD.relative_to(REPO_ROOT))}

{'card': 'research/dashboard_failure_modes_v1',
 'source': 'research/analysis_failure_modes_v1/analysis_report'}

## 读取 manifest-first dashboard payload

输入来自 `analysis_failure_modes_v1/analysis_report/analysis_manifest.json`，dashboard 只能通过 manifest/export 读取公开 sidecar。

In [2]:
payload = load_model_matrix_dashboard(SOURCE_CARD, top_n=5)
assert payload['manifest_source']['kind'] == 'analysis_manifest'
assert payload['failure_modes']
assert 'failure_modes' in payload['artifacts']
(CARD / 'dashboard_export.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
payload['failure_modes'][:2]

[{'schema_version': 'analysis.failure_modes.v1',
  'model': 'gcn_mse',
  'baseline_model': 'ridge_mse',
  'segment_type': 'regime',
  'segment': 'stress',
  'failure_mode': 'return_stability_lag',
  'severity_score': 78.04619,
  'relative_return_pct_delta': -3.9118617744127393,
  'stability_score_delta': -68.27138855380323,
  'max_drawdown_pct_delta': 3.90215492137449,
  'turnover_delta': 0.0,
  'diagnosis': '弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。',
  'review_note': 'gcn_mse 在 stress 同时落后基准 ridge_mse 的相对收益和稳定性，优先检查该 regime/window 的信号适配。'},
 {'schema_version': 'analysis.failure_modes.v1',
  'model': 'gcn_mse',
  'baseline_model': 'ridge_mse',
  'segment_type': 'window',
  'segment': 'stress_window',
  'failure_mode': 'return_stability_lag',
  'severity_score': 78.04619,
  'relative_return_pct_delta': -3.9118617744127393,
  'stability_score_delta': -68.27138855380323,
  'max_drawdown_pct_delta': 3.90215492137449,
  'turnover_delta': 0.0,
  'diagnosis': '弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 

## 构建 view model 与静态页面

view model 应包含全局 failure summary，模型卡片也应包含匹配模型和 segment 的 failure modes。

In [3]:
view_model = build_dashboard_view_model(payload)
assert view_model['failure_mode_summary']['count'] == len(payload['failure_modes'])
assert view_model['failure_mode_summary']['top_failure_mode'] == 'return_stability_lag'
assert any(card.get('failure_modes') for card in view_model['model_cards'])
(CARD / 'view_model.json').write_text(json.dumps(view_model, ensure_ascii=False, indent=2), encoding='utf-8')
html_result = write_dashboard_html(view_model, output_path=CARD / 'dashboard.html', title='SOXL Failure Modes Dashboard')
html_text = (CARD / 'dashboard.html').read_text(encoding='utf-8')
assert 'Failure Modes' in html_text
assert 'return_stability_lag' in html_text
assert 'failure_modes.csv' not in html_text
view_model['failure_mode_summary']

{'count': 4,
 'top_failure_mode': 'return_stability_lag',
 'top_model': 'gcn_mse',
 'top_segment': 'stress',
 'top_severity_score': 78.04619}

In [4]:
summary = {
    'schema_version': 'research.dashboard_failure_modes_v1',
    'status': 'completed',
    'source_card': str(SOURCE_CARD.relative_to(REPO_ROOT)),
    'dashboard_path': str((CARD / 'dashboard.html').relative_to(REPO_ROOT)),
    'view_model_path': str((CARD / 'view_model.json').relative_to(REPO_ROOT)),
    'failure_mode_count': view_model['failure_mode_summary']['count'],
    'top_failure_mode': view_model['failure_mode_summary']['top_failure_mode'],
    'top_failure_segment': view_model['failure_mode_summary']['top_segment'],
    'sidecar_paths_hidden': 'failure_modes.csv' not in html_text,
    'conclusion': 'dashboard 已能从 manifest-first analysis report 读取并渲染 failure_modes，而不暴露 sidecar 路径。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'research.dashboard_failure_modes_v1',
 'status': 'completed',
 'source_card': 'research/analysis_failure_modes_v1/analysis_report',
 'dashboard_path': 'research/dashboard_failure_modes_v1/dashboard.html',
 'view_model_path': 'research/dashboard_failure_modes_v1/view_model.json',
 'failure_mode_count': 4,
 'top_failure_mode': 'return_stability_lag',
 'top_failure_segment': 'stress',
 'sidecar_paths_hidden': True,
 'conclusion': 'dashboard 已能从 manifest-first analysis report 读取并渲染 failure_modes，而不暴露 sidecar 路径。'}

## 中文结论

- 做了什么：验证 dashboard 从 manifest-first analysis report 读取 `failure_modes`，并在 view model 与静态 HTML 中展示 Failure Modes 区块。
- 核心指标：failure mode 数量与 source payload 一致，top failure 为 `return_stability_lag`，页面不暴露 `failure_modes.csv` 路径。
- regime 差异：最高优先级失败模式指向 stress window，dashboard 可以把压力期收益/稳定性同步落后作为 review 入口。
- 是否晋升：这是 dashboard 框架展示能力，可保留在 `src/dashboard`；不是策略晋升。
- 风险：页面当前只展示模型卡片内的前几条 failure modes，全局摘要还没有单独视觉卡片。
- 下一步：可以把 failure mode summary 提升为 dashboard 顶部独立 panel，并接入 schedule 通知摘要。